# K-Fold - Using hand written digits dataset

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.datasets import load_digits

In [2]:
digits = load_digits()

In [3]:
# Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.3)

In [4]:
# Fit Logistic Regression
lor = LogisticRegression()
lor.fit(X_train, y_train)
lor.score(X_test, y_test)

D:\Complete DataScience\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9722222222222222

In [5]:
# SVM
svm = SVC()
svm.fit(X_train, y_train)
svm.score(X_test, y_test)

0.987037037037037

In [6]:
# Random FOrest
rf = RandomForestClassifier(n_estimators=60)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

0.9685185185185186

## KFold

In [7]:
from sklearn.model_selection import KFold

# This divides the data into 4 folds.
# Behind the scenes, the dataset is split into 4 parts. On each iteration, 3 folds are used for training, and 1 fold is used for validation/testing. This process repeats 4 times, so every fold gets to be the validation set once
kf = KFold(n_splits=4)
kf

KFold(n_splits=4, random_state=None, shuffle=False)

In [8]:
# [i for  i in range(1, 10)] --> I am just using this a dummy dataset (9 samples -- 1 to 9). They dont actually matter, its just to see how KFold splits the indices

# kf.split is a generator that yields two arrays each time.
# 1) Train_index --> indices of the training samples for that fold
# 2) test_index --> indices of the testing samples for that fold
for train_index, test_index in kf.split([i for  i in range(1, 10)]):
  print(train_index, test_index)

[3 4 5 6 7 8] [0 1 2]
[0 1 2 5 6 7 8] [3 4]
[0 1 2 3 4 7 8] [5 6]
[0 1 2 3 4 5 6] [7 8]


In [9]:
# This defines a function named get_score(). This function will train the given model and return its performance score on the test set
# Inputs: model (ML model), X_train(training features), X_test(testing features), y_train(labels for the training set), y_test(labels for the test set).

def get_score(model, X_train, X_test, y_train, y_test):
  model.fit(X_train, y_train)
  return model.score(X_test, y_test)

In [10]:
get_score(svm, X_train, X_test, y_train, y_test)

0.987037037037037

## StratifiedKFold

In [11]:
from sklearn.model_selection import StratifiedKFold
# This creates a stratifiedKfold object that divides my dataset into 3 folds
folds = StratifiedKFold(n_splits=3)

# When i call folds.split(X, y), it needs both X(features) and y(labels) - unlike basic KFold which only needed X

In [12]:
# Three empty lists - one for each model. After the loop, these lists will contain one score per fold
scores_l = []
scores_svm = []
scores_rf = []

# Cross Validation loop

# digits.data --> the feature matrix
# digits.target --> the true digit labels(0-9)

# This takes the indices  provided by folds.split() and uses them to slice the dataset into X_train, X_test, y_train, y_test
for train_index, test_index in folds.split(digits.data,digits.target):
    X_train, X_test, y_train, y_test = digits.data[train_index], digits.data[test_index], digits.target[train_index], digits.target[test_index]
    scores_l.append(get_score(LogisticRegression(solver='liblinear',multi_class='ovr'), X_train, X_test, y_train, y_test))
    scores_svm.append(get_score(SVC(gamma='auto'), X_train, X_test, y_train, y_test))
    scores_rf.append(get_score(RandomForestClassifier(n_estimators=40), X_train, X_test, y_train, y_test))

D:\Complete DataScience\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
D:\Complete DataScience\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
D:\Complete DataScience\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
D:\Complete DataSc

In [13]:
scores_l

[0.8948247078464107, 0.9532554257095158, 0.9098497495826378]

In [14]:
scores_svm

[0.3806343906510851, 0.41068447412353926, 0.5125208681135225]

In [15]:
scores_rf

[0.9432387312186978, 0.9532554257095158, 0.9232053422370617]

## Cross_Val Score Function

In [16]:
# This is a built-in utility for performing cross_validation automatically so you dont have to manually create KFold or stratifiedKFold objects, split the data yourself, train test your model inside the loop
from sklearn.model_selection import cross_val_score

In [17]:
# Logistic Regression model performance using cross_val_score

# The model I am using is Logistic Rgeression
cross_val_score(LogisticRegression(solver='liblinear',multi_class='ovr'), digits.data, digits.target,cv=3, n_jobs=-1)
# The output is an array of scores, one for each fold. Each number is the models accurcy on one folds test data

array([0.89482471, 0.95325543, 0.90984975])

In [18]:
# SVM model performance using cross val_score
cross_val_score(SVC(gamma='auto'), digits.data, digits.target,cv=3)

array([0.38063439, 0.41068447, 0.51252087])

In [19]:
# Random Forest performance using crossvalscore
cross_val_score(RandomForestClassifier(n_estimators=40),digits.data, digits.target,cv=3)

array([0.93489149, 0.94490818, 0.92654424])